# DigiPico - Gemma 4 Fine-Tuning with Unsloth
This notebook is designed to run on a DigitalOcean GPU Droplet.
It loads the `sentence-transformers/eli5` dataset, formats it for instruction tuning, and fine-tunes Gemma 4 using Unsloth for maximum speed and VRAM efficiency.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# Note: Replace 'google/gemma-4-8b-it' with the exact HuggingFace model ID when available
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-7b-it-bnb-4bit", # Using Gemma 7B as a placeholder for Gemma 4
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### Load and Format the ELI5 Dataset

In [ ]:
from datasets import load_dataset

# Load the dataset as requested
ds = load_dataset("sentence-transformers/eli5")

prompt_template = """<bos><start_of_turn>user
Explain this simply: {}
<end_of_turn>
<start_of_turn>model
{}
<end_of_turn>"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    questions = examples["title"]
    # The ELI5 dataset usually has multiple answers, we'll take the first one (highest scored)
    answers = [a[0] if isinstance(a, list) and len(a) > 0 else str(a) for a in examples["answers"]]
    
    texts = []
    for question, answer in zip(questions, answers):
        text = prompt_template.format(question, answer) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Take a small subset for quick fine-tuning (e.g. 5000 rows)
train_dataset = ds['train'].select(range(5000))
mapped_dataset = train_dataset.map(formatting_prompts_func, batched = True, remove_columns=train_dataset.column_names)

### Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = mapped_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### Save the Model (GGUF format for Ollama/Local inference)

In [ ]:
# Save LoRA adapters
model.save_pretrained("digipico_gemma_lora")
tokenizer.save_pretrained("digipico_gemma_lora")

# Export to GGUF format so we can load it into Ollama locally!
# Choose q4_k_m for a good balance of size and quality
model.save_pretrained_gguf("digipico_gemma_gguf", tokenizer, quantization_method = "q4_k_m")